# Ad A vs Ad B: the pilot, 600 visitors per ad

**Hypothesis Testing for the Difference in the Population Proportions** · a worked example from the StatNex teaching guide

**The story.** Before a big A/B test, an online shop ran a small pilot: 600 visitors saw Ad A (the current banner) and another 600 saw Ad B (a new video ad). The video costs \$2,000 a month more to run, so the team wants an early sign of whether it is worth it. It counted the share who clicked.

**The question.** Does Ad B get a higher click-through rate than Ad A?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `03_two_sample/two_sample_proportion_test__pilot.csv` (2 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Variable,Ad A,Ad B
Sample size,600.0,600.0
Click-through rate,0.08,0.115

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import (confint_proportions_2indep, proportion_effectsize,
                                          proportions_ztest)

pd.set_option("display.width", 120)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
# Each column holds the sample size and the proportion, top to bottom.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
samples1 = ["Ad A"]
samples2 = ["Ad B"]
confidence = 95 / 100
z_critical = stats.norm.ppf(0.5 + confidence / 2)

summary = {}
for name in [*samples1, *samples2]:
    size, p = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)[:2]
    summary[name] = (int(round(size)), p)
descriptive = pd.DataFrame.from_dict(summary, orient="index", columns=["n", "proportion"])
print(descriptive)

## Difference of proportions: interval and z test

In [ ]:
# Every column of sample 1 is compared with every column of sample 2. The z test asks:
# if both groups had the same true rate, how often would chance produce a gap this
# large? It uses the pooled proportion, the variance the null hypothesis implies
# (statsmodels' proportions_ztest). The interval (Wald's) uses each sample's own
# variance instead. Both assume two independent random samples, large enough that
# each holds at least about 5 "yes" and 5 "no" answers.
intervals, tests = [], []
for a in samples1:
    for b in samples2:
        (n1, p1), (n2, p2) = summary[a], summary[b]
        difference = p1 - p2
        # The counts n p need not be whole numbers here: a rounded proportion still works.
        z, p_two = proportions_ztest([n1 * p1, n2 * p2], [n1, n2])
        # Use the one-sided P only if its direction was fixed before the test started.
        p_one = p_two / 2
        lower, upper = confint_proportions_2indep(n1 * p1, n1, n2 * p2, n2, method="wald",
                                                  alpha=1 - confidence)
        intervals.append([a, b, difference, lower, upper])
        tests.append([a, b, z, p_two, stars(p_two), p_one, stars(p_one)])

intervals = pd.DataFrame(intervals, columns=["sample 1", "sample 2", "difference", "lower", "upper"])
tests = pd.DataFrame(tests, columns=["sample 1", "sample 2", "z", "P two-sided", "",
                                     "P one-sided", ""])
print(f"\n{confidence:.0%} confidence interval for the difference")
print(intervals.to_string(index=False))
print()
print(tests.to_string(index=False))

## Effect sizes, with confidence intervals

In [ ]:
# The risk difference is the difference above, now with Newcombe's (1998) method-10
# interval, which statsmodels calls "newcomb": each proportion's Wilson score interval
# (l, u), combined as d - sqrt((p1 - l1)² + (u2 - p2)²) and d + sqrt((u1 - p1)² +
# (p2 - l2)²). Its coverage is closer to the stated level than Wald's, and it never
# leaves [-1, 1]. Cohen's h = 2 asin sqrt(p1) - 2 asin sqrt(p2) (statsmodels'
# proportion_effectsize) reads on one scale whatever the proportions' size;
# 2 asin sqrt(p) has variance 1/n, so its interval is h ± z sqrt(1/n1 + 1/n2)
# (Cohen 1988). For click or conversion rates, the risk difference in points, and the
# relative lift, usually say more to a manager than h does.
def size(value):
    """Cohen's (1988) reading of |h|: 0.2 small, 0.5 medium, 0.8 large."""
    return ["negligible", "small", "medium", "large"][sum(abs(value) >= s for s in (0.2, 0.5, 0.8))]


rows = []
for a in samples1:
    for b in samples2:
        (n1, p1), (n2, p2) = summary[a], summary[b]
        difference = p1 - p2
        lower, upper = confint_proportions_2indep(n1 * p1, n1, n2 * p2, n2, method="newcomb",
                                                  alpha=1 - confidence)
        rows.append([a, b, "risk difference", difference, lower, upper, ""])
        h = proportion_effectsize(p1, p2)
        half = z_critical * np.sqrt(1 / n1 + 1 / n2)
        rows.append([a, b, "Cohen's h", h, h - half, h + half, size(h)])
effect_sizes = pd.DataFrame(rows, columns=["sample 1", "sample 2", "measure", "value", "lower",
                                           "upper", "size"])
print()
print(effect_sizes.to_string(index=False))

## What the result says

In the pilot, 8.0% of visitors clicked Ad A and 11.5% clicked Ad B. The difference of proportions (Ad A minus Ad B) is -0.035. The minus sign means that Ad B is ahead.

The z statistic is -2.04 and the two-sided P value is 0.041. The P value is the chance of a gap at least this large if both ads were equally good. It is small, so Ad B is ahead, but only just.

The 95% confidence interval, the range of values the data agree with for the true difference, runs from -0.069 to -0.0015. So the true gain could be almost nothing or very large. The pilot gives an early sign but cannot pin down how much better Ad B is. This is why the shop went on to a much larger test.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.